# 机器学习概述（下）：`Runner` 类与加州房价案例

上一节我们用线性回归走通了机器学习实践的五要素，但每次重复写训练循环、评估、保存比较烦琐。本节做两件事：

1. 抽象一个 **`Runner`** 类，把训练循环、评估、保存/加载封装起来——后续章节会持续复用并扩展它；
2. 跑通**加州房价回归**端到端案例：从 sklearn 拉数据 → 训练/测试划分 → 特征归一化 → 训练 → 评估 → 预测。

> **关于数据集选择**：本节用 sklearn 内置的加州房价数据集（`fetch_california_housing`）。原版书使用的是波士顿房价数据集，但 sklearn 1.2 起已经因伦理问题移除了波士顿数据集，加州房价是目前推荐的回归示范数据集——样本量更大（20640 个），没有缺失值，特征清晰。


## 1. `Runner`：训练 / 评估 / 保存 / 加载 的最小封装

在一个任务上应用机器学习方法的流程基本上包括数据集构建、模型构建、损失函数定义、优化器定义、评价指标定义、模型训练、模型评价以及模型预测等环节。为了把上述环节规范化，我们把机器学习模型的基本要素封装成一个 `Runner` 类。

`Runner` 类约定如下成员函数：

- **`__init__`**：实例化 `Runner` 时默认调用，传入模型、损失函数、优化器和评价指标等；
- **`fit`**：完成模型训练，指定训练集和验证集；
- **`evaluate`**：通过对训练好的模型进行评价，查看模型训练效果；
- **`predict`**：选取一条数据对训练好的模型进行预测；
- **`save`**：在训练过程和训练结束后保存模型；
- **`load`**：加载之前保存的模型。

### `Runner` 的三个版本

本书会循序渐进地实现 3 个版本的 `Runner`：

| 版本 | 关键能力 | 引入章节 |
|---|---|---|
| `RunnerV1` | 直接调最小二乘法解析解；只做保存 | 本节末尾 |
| `RunnerV2` | 加入梯度下降法训练；记录 train/dev 损失；保存最优模型 | 第 3 章 |
| `RunnerV3` | 用 `DataLoader` 做小批量训练；`state_dict` 保存 | 第 4 章 |

`RunnerV3` 已经够用于绝大多数任务，后续章节都基于它做轻量扩展。本节用一个稍微高级的写法（小批量训练 + `state_dict`，相当于一个简化版 V3），让加州房价案例直接跑得起来。


In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


class Runner:
    """训练 / 评估 / 保存 / 加载 的最小封装。

    后续章节会扩展：early stopping、metrics 记录、学习率调度等。
    """

    def __init__(self, model, optimizer, loss_fn, metric_fn=None):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.metric_fn = metric_fn      # 缺省时用 loss 当指标
        self.history = {'train_loss': [], 'dev_metric': []}

    def fit(self, train_loader, dev_loader=None, num_epochs=100, log_every=10):
        for epoch in range(num_epochs):
            self.model.train()
            running = 0.0
            for x, y in train_loader:
                self.optimizer.zero_grad()
                loss = self.loss_fn(self.model(x), y)
                loss.backward()
                self.optimizer.step()
                running += loss.item() * x.size(0)
            train_loss = running / len(train_loader.dataset)
            self.history['train_loss'].append(train_loss)

            if dev_loader is not None:
                dev_m = self.evaluate(dev_loader)
                self.history['dev_metric'].append(dev_m)
                if (epoch + 1) % log_every == 0:
                    print(f'epoch {epoch+1:4d}  train_loss={train_loss:.4f}  dev_metric={dev_m:.4f}')
            elif (epoch + 1) % log_every == 0:
                print(f'epoch {epoch+1:4d}  train_loss={train_loss:.4f}')

    @torch.no_grad()
    def evaluate(self, loader):
        self.model.eval()
        total, n = 0.0, 0
        fn = self.metric_fn if self.metric_fn is not None else self.loss_fn
        for x, y in loader:
            total += fn(self.model(x), y).item() * x.size(0)
            n += x.size(0)
        return total / n

    @torch.no_grad()
    def predict(self, x):
        self.model.eval()
        return self.model(x)

    def save(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.model.state_dict(), path)

    def load(self, path):
        self.model.load_state_dict(torch.load(path, weights_only=True))


## 2. 加州房价回归

加州房价数据集来自 sklearn 自带的 `fetch_california_housing`：

- 样本数：20 640
- 特征数：8 个（中位收入、房龄、平均房间数、人口、地理位置等）
- 目标：街区房价中位数（单位 $100k）

实践流程包括如下步骤：

1. **数据处理**：加载数据、查看缺失值、划分训练/测试集、特征归一化；
2. **模型构建**：定义线性回归模型；
3. **训练 + 评估**：用 `Runner` 跑通；
4. **保存与加载**：演示 `state_dict` 的存取；
5. **检查权重 + 单点预测**：解释模型学到了什么。


### 2.1 加载数据

通过 sklearn 直接拉取 DataFrame，避免本地额外的 CSV 文件依赖。


In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd

ds = fetch_california_housing(as_frame=True)
df = ds.frame                # DataFrame: 8 特征 + 1 target (MedHouseVal)
print('shape:', df.shape)
print('missing per column:', df.isna().sum().sum(), '(no NaN)')
df.head()


**字段说明**：

| 列 | 含义 |
|---|---|
| `MedInc` | 街区中位家庭收入（单位 $10k） |
| `HouseAge` | 街区房屋中位年龄 |
| `AveRooms` | 平均房间数 |
| `AveBedrms` | 平均卧室数 |
| `Population` | 街区人口 |
| `AveOccup` | 户均人口 |
| `Latitude` / `Longitude` | 街区地理坐标 |
| `MedHouseVal` | **目标**：房价中位数（$100k） |

从输出看，加州房价数据集没有缺失值，可以直接进入下一步。


### 2.2 数据划分 + 特征归一化

由于度量单位不同，不同特征值的取值范围差别很大（例如 `MedInc` 在 $0\sim15$，`Population` 在 $1\sim35\,000$）。为了消除这种影响，在模型训练前要对特征数据进行**特征缩放**（feature scaling），使不同特征之间具有可比性。最常见的两种是：

- **归一化**（normalization）：把每一维特征缩放到 $[0, 1]$；
- **标准化**（standardization）：把每一维特征改造为均值 0、方差 1。

这里用标准化，对深度学习模型一般更友好。

> **关键约定**：归一化的 `mean/std` **只能在训练集上拟合**，再用同一组统计量去变换测试集——否则会发生测试集信息泄露。


In [ ]:
torch.manual_seed(0)

X = torch.tensor(df.drop(columns=['MedHouseVal']).values, dtype=torch.float32)
y = torch.tensor(df['MedHouseVal'].values, dtype=torch.float32).unsqueeze(1)

# 8:2 划分训练 / 测试集
N = len(X)
perm = torch.randperm(N)
split = int(N * 0.8)
tr, te = perm[:split], perm[split:]
X_train, y_train = X[tr], y[tr]
X_test,  y_test  = X[te], y[te]

# mean/std 只在训练集上拟合
mean, std = X_train.mean(dim=0), X_train.std(dim=0)
X_train = (X_train - mean) / std
X_test  = (X_test  - mean) / std

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=128)
print(f'train: {len(X_train)}  test: {len(X_test)}  feature dim: {X_train.shape[1]}')


### 2.3 模型 + `Runner` + 训练

线性回归模型 = `nn.Linear(8, 1)`。用 MSE 作为损失和评估指标。


In [ ]:
torch.manual_seed(0)

model = nn.Linear(X_train.shape[1], 1)
optimizer = optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

runner = Runner(model, optimizer, loss_fn)
runner.fit(train_loader, dev_loader=test_loader, num_epochs=50, log_every=10)


### 2.4 保存与加载

用 `Path` 拼绝对路径，避免 notebook 在不同当前目录下表现不一致。**推荐保存 `state_dict`** 而不是整个 `model` 对象——前者跨结构改动更稳。


In [ ]:
MODEL_DIR = Path.cwd() / 'models'
save_path = MODEL_DIR / 'california_linear.pt'
runner.save(save_path)

# 新建一个模型，从磁盘加载参数，验证 MSE 与原模型一致
model2 = nn.Linear(X_train.shape[1], 1)
runner2 = Runner(model2, optim.Adam(model2.parameters()), loss_fn)
runner2.load(save_path)
print('test MSE (reloaded):', runner2.evaluate(test_loader))


### 2.5 检查学到的权重

特征做过标准化之后，权重的正负就能定性解释——负权重表示"该特征增大 → 房价下降"。


In [ ]:
feat_names = list(df.drop(columns=['MedHouseVal']).columns)
weights = model.weight.detach().squeeze().tolist()
bias = model.bias.item()

for name, w in sorted(zip(feat_names, weights), key=lambda kv: kv[1]):
    print(f'{name:12s}  {w:+.4f}')
print(f'\nbias         {bias:+.4f}')


**典型观察**：

- `MedInc`（中位收入）权重最正——收入越高的街区房价越高；
- `Latitude` 权重为负，反映加州北部（纬度高）房价低于南部沿海；
- `AveRooms` 权重也呈正——平均房间数大暗示是低密度高端街区。


### 2.6 单点预测


In [ ]:
x0, y0 = X_test[:1], y_test[:1]
with torch.no_grad():
    pred = runner.predict(x0)
print(f'true price (×$100k): {y0.item():.2f}  →  pred: {pred.item():.2f}')


### 2.7 训练曲线


In [ ]:
import matplotlib.pyplot as plt

plt.plot(runner.history['train_loss'], label='train loss')
plt.plot(runner.history['dev_metric'], label='test MSE')
plt.xlabel('epoch'); plt.ylabel('MSE'); plt.legend(); plt.grid(alpha=0.3); plt.show()


## 小结

本节抽象了机器学习训练的样板代码，并端到端跑通了加州房价回归：

- **`Runner`** 把训练循环、评估、模型 IO 封装成一个对象——后续章节会扩展（早停、metric 记录、学习率调度、混合精度等）；
- **特征归一化** 的统计量只在训练集上拟合，再变换测试集，避免信息泄露；
- **`state_dict` 保存 / 加载** 是 PyTorch 官方推荐的做法——比直接 pickle 整个模型更稳；
- **数据集选择**：加州房价是当前 sklearn 推荐的回归示范数据集，波士顿房价已在 sklearn 1.2 中移除。

在本章中，我们通过实现线性回归对机器学习的实践有了初步了解，并介绍了实践机器学习的五个基本要素：数据、模型、学习准则（损失函数）、优化算法和评价指标。在后续章节中，我们会不断完善 `Runner` 类和增加更多算子。
